# Message Authentication Code (MAC) tag

**Task**: Alice and Bob previously agreed on a secret key. Alice sent an encrypted message to Bob. Bob wants to be sure that the message really came from Alice and not from someone else. How does Bob *authenticate* Alice's encrypted message?

**Definition**: Message Authentication Code
- $\mathtt{Gen}$: generates a secret $k$ key with uniform distribution
- $\mathtt{Mac}$: creates a $t$ *tag* from the $m$ message, which we attach to the encrypted message: $t \leftarrow \mathtt{Mac}(k, m)$
- $\mathtt{Vrfy}$: given $k$ secret key, $m$ message and $t$ tag. The output is $b=1$ if the tag is good, otherwise $b=0$: $b := \mathtt{Vrfy}(k, m, t)$

Requirement: $\mathtt{Vrfy}(k, m, \mathtt{Mac}(k, m)) = 1$

## CBC-MAC

![CBC-MAC](./cbc-mac.png)

In [0]:
import secrets
from typing import Optional

from des import DES, DES_CBC
from utils import xor_strings


class DES_CBC_MAC(DES):
    def __init__(self, key, block_size: int = 8, iv: Optional[bytes] = None) -> None:
        super().__init__(key, block_size)
        self.iv = iv or secrets.token_bytes(block_size)

    def mac(self, message: bytes) -> bytes:
        if len(message) % self.block_size != 0:
            raise ValueError("Message is not a multiple of block size.")
        res = self.iv
        for b in range(0, len(message), self.block_size):
            msg_block = message[b : b + self.block_size]
            x = xor_strings(msg_block, res)
            res = self._encryption(x, True)
        return self.iv + res

    def vrfy(self, message: bytes, tag: bytes) -> bool:
        if len(message) % self.block_size != 0:
            return False
        return tag == self.mac(message)

In [0]:
message = b"cryptography0123"

In [0]:
mac_master = DES_CBC_MAC(b"animator")
tag_master = mac_master.mac(message)
tag_master

In [0]:
des = DES_CBC(b"animator")
des.encrypt(message)

**Question**: What could be the problem with the above construction?

**Solution**: There are two main problems:
1. The IV is random
    - Flip some bits of $m_1$ ($m_1'$), and the same bits of $IV_1$ where we modified $m_1$ ($IV_1'$).
    - then $c_1' = F(k, m_1' \oplus IV_1')$
    - since we modified the same bits, the changes cancel out in the first application of $F$, i.e., $m_1' = m_1$
    - this attack only works on the first block
2. The same key is used for generating the MAC tag and for encryption
    - Assume there is no padding check
    - Eve can modify any ciphertext block except the last one: $C' = C_1'C_2'\cdots C_n$
    - Bob receives $C'$ and decrypts it, getting a $P'$ plaintext
        - $P_n'$ differs from $P_n$, but $C_n$ is the same: $P_n' = C_{n-1}' \oplus D(C_n)$
    - after decrypting Bob calculates the tag: $t' = E(P_n' \oplus E(P_{n-1}' \oplus E(\cdots \oplus E(P_1')))) = E(P_n' \oplus C_{n-1}')$
    - but the $t'$ tag exactly matches $C_n$: $t' = E(C_{n-1}' \oplus D(C_n) \oplus C_{n-1}') = E(D(C_n)) = C_n$
    - i.e., $t' = C_n = t$

Demonstration of the first problem:

In [0]:
def invert_byte(b):
    if not isinstance(b, int):
        b = ord(b)
    return bytes.fromhex(f"{b ^ 0xFF:02x}")

In [0]:
message_ = invert_byte(message[0]) + message[1:]
message, message_

In [0]:
iv_ = invert_byte(mac_master.iv[0]) + mac_master.iv[1:]
mac_master.iv, iv_

In [0]:
mac2 = DES_CBC_MAC(b"animator", iv=iv_)
tag_ = mac2.mac(message_)
print(tag_master)
print(tag_)

In [0]:
tag_master[8:] == tag_[8:]

Demonstration of the second problem:

In [0]:
des = DES_CBC(b"animator", iv=bytes(8))
ct = des.encrypt(message, False)
ct

In [0]:
mac = DES_CBC_MAC(b"animator", iv=bytes(8))
tag = mac.mac(message)
tag

In [0]:
ct_ = ct[:8] + secrets.token_bytes(8) + ct[16:]
pt_ = des.decrypt(ct_, False)
pt_

In [0]:
mac.vrfy(pt_, tag)

In [0]:
mac2 = DES_CBC_MAC(secrets.token_bytes(8), iv=bytes(8))
tag2 = mac2.mac(message)
tag2

In [0]:
mac2.vrfy(pt_, tag2)

**Question**: Let $m$ be a message exactly one block long. How can a message twice as long have the same tag?

**Solution**: Let $m$ be a one-block long message, whose tag is $t = F(k, m)$. Let $m' = m\;||\;(m \oplus t)$. Then $t_1 = F(k, m) = t$ and $t_2 = F(k, t \oplus (t \oplus m)) = F(k, t \oplus t \oplus m) = F(k, m) = t$, i.e., $t_1 = t_2$

In [0]:
message = b"humanoid"

In [0]:
mac = DES_CBC_MAC(secrets.token_bytes(8), iv=bytes(8))
tag = mac.mac(message)
tag

In [0]:
message2 = message + xor_strings(message, tag[8:])
message2

In [0]:
mac2 = DES_CBC_MAC(mac.key, iv=mac.iv)
tag2 = mac2.mac(message2)
tag2

In [0]:
tag

Correction could be: put the length of the message in front of the message, i.e., $m' = |m| + m$ and calculate the tag of this

## Authenticated encryption

Let $k_E$ and $k_M$ be independent keys, let $m$ be a message.

- *Encrypt-and-authenticate*: encryption and authentication are independent: $c \leftarrow \mathtt{Enc}(k_E, m) \quad t \leftarrow \mathtt{Mac}(k_M, m)$
    - It may not be CPA-secure: if we use CBC-MAC, the same message will get the same tag
- *Authenticate-then-encrypt*: first we calculate the MAC tag, then we encrypt the tag and the $m$ message together: $t \leftarrow \mathtt{Mac}(k_M, m) \quad c \leftarrow \mathtt{Enc}(k_E, m||t)$
    - There can be two errors when decrypting: `BadPaddingError` or invalid tag
    - If the two exceptions are distinguishable, padding-oracle attack can be applied
    - IPsec attack in 2010
    - SSL: indistinguishable exceptions, but was breakable with side-channel attack
- *Encrypt-then-authenticate*: $c \leftarrow \mathtt{Enc}(k_E, m) \quad t \leftarrow \mathtt{Mac}(k_M, c)$
    - We first verify a secret text, then decrypt it
    - Resistant to padding-oracle attack

**Question**: Why do $k_E$ and $k_M$ need to be independent of each other?

**Solution**: Because if we use the same key, we get the plaintext back:
- Let $\mathtt{Enc}(k, m) = F(k, m||r)$ and $\mathtt{Mac}(k, c) = F^{-1}(k, c)$, where $m \in \{0,1\}^{n/2}, r \in \{0,1\}^{n/2}$
- $\mathtt{Enc}(k, m) = F(k, m||r)$
- $\mathtt{Mac}(\mathtt{Enc}(k, m||r)) = F^{-1}(k, F(k, m||r)) = m||r$